# 01 — Exploratory Data Analysis & Visualization

**Thesis:** Macroeconomic Factor-Based Dynamic Portfolio Optimization

This notebook provides a comprehensive exploration of the raw data:
1. Asset price data (10 ETFs spanning major asset classes)
2. Macroeconomic indicators from FRED
3. Return distributions & summary statistics
4. Correlation structure
5. Stationarity analysis
6. Rolling volatility & regime detection

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from src.config import (
    RAW_DIR, PROCESSED_DIR, RESULTS_DIR,
    TICKERS, FRED_SERIES, TICKER_LIST,
    START_DATE, END_DATE, STRESS_SCENARIOS,
)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.figsize': (14, 6), 'figure.dpi': 100})

print('Configuration loaded.')
print(f'Date range: {START_DATE} to {END_DATE}')
print(f'Assets: {TICKER_LIST}')

## 1. Load Data

In [ ]:
# Load or fetch data
try:
    prices = pd.read_csv(RAW_DIR / 'prices.csv', index_col=0, parse_dates=True)
    macro = pd.read_csv(RAW_DIR / 'macro.csv', index_col=0, parse_dates=True)
    print(f'Loaded prices: {prices.shape}')
    print(f'Loaded macro: {macro.shape}')
except FileNotFoundError:
    from src.data.fetcher import fetch_all
    prices, macro = fetch_all()

# Daily returns
returns = prices.pct_change().dropna()
print(f'Daily returns: {returns.shape}')
print(f'Date range: {returns.index[0].date()} to {returns.index[-1].date()}')

## 2. Price Time Series

In [ ]:
# Normalized price series (base = 100)
normalized = prices / prices.iloc[0] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: all assets
for col in normalized.columns:
    axes[0].plot(normalized.index, normalized[col], label=f'{col} ({TICKERS[col]})', linewidth=1.0)
axes[0].set_title('Normalized Price Series (Base = 100)')
axes[0].set_ylabel('Indexed Value')
axes[0].legend(fontsize=8, loc='upper left')
axes[0].set_yscale('log')

# Right: by asset class
from src.config import ASSET_CLASSES
colors = {'Equity': 'tab:blue', 'Fixed Income': 'tab:green', 'Alternatives': 'tab:orange'}
for cls, tickers in ASSET_CLASSES.items():
    for t in tickers:
        if t in normalized.columns:
            axes[1].plot(normalized.index, normalized[t], label=t, color=colors[cls], alpha=0.7, linewidth=1.0)
axes[1].set_title('Price Series by Asset Class')
axes[1].set_ylabel('Indexed Value')
axes[1].legend(fontsize=8, ncol=2)
axes[1].set_yscale('log')

# Shade stress periods
for ax in axes:
    for name, (start, end) in STRESS_SCENARIOS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color='red')

plt.tight_layout()
plt.show()

## 3. Summary Statistics

In [ ]:
# Annualized summary statistics
summary = pd.DataFrame({
    'Asset Class': [TICKERS[t] for t in returns.columns],
    'Ann. Return': returns.mean() * 252,
    'Ann. Volatility': returns.std() * np.sqrt(252),
    'Sharpe Ratio': (returns.mean() * 252) / (returns.std() * np.sqrt(252)),
    'Skewness': returns.skew(),
    'Kurtosis': returns.kurtosis(),
    'Max Drawdown': pd.Series({
        col: ((prices[col] / prices[col].cummax()) - 1).min()
        for col in prices.columns
    }),
    'Best Day': returns.max(),
    'Worst Day': returns.min(),
})

# Format for display
display_cols = {
    'Ann. Return': '{:.2%}', 'Ann. Volatility': '{:.2%}',
    'Sharpe Ratio': '{:.3f}', 'Skewness': '{:.3f}',
    'Kurtosis': '{:.2f}', 'Max Drawdown': '{:.2%}',
    'Best Day': '{:.2%}', 'Worst Day': '{:.2%}',
}
styled = summary.style.format(display_cols)
styled

## 4. Return Distributions

In [ ]:
# Return distribution plots
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(returns.columns):
    ax = axes[i]
    data = returns[col].dropna()
    
    # Histogram with KDE
    ax.hist(data, bins=80, density=True, alpha=0.6, color='steelblue', edgecolor='none')
    
    # Overlay normal distribution
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, data.mean(), data.std()), 'r--', linewidth=1.5, label='Normal')
    
    ax.set_title(f'{col}', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(labelsize=8)
    
    # Jarque-Bera test
    jb_stat, jb_p = stats.jarque_bera(data)
    ax.text(0.02, 0.95, f'JB p={jb_p:.2e}', transform=ax.transAxes, fontsize=7, va='top')

plt.suptitle('Daily Return Distributions vs Normal', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# QQ plots for normality assessment
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(returns.columns):
    stats.probplot(returns[col].dropna(), dist='norm', plot=axes[i])
    axes[i].set_title(f'{col}', fontsize=10)
    axes[i].get_lines()[0].set_markersize(2)

plt.suptitle('QQ Plots: Daily Returns vs Normal Distribution', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Full-period correlation matrix
corr = returns.corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], square=True, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Full-Period Correlation Matrix')

# Clustered heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], square=True, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlation Matrix (Full)')

plt.tight_layout()
plt.show()

In [ ]:
# Rolling correlation between SPY and other assets
window = 63  # ~3 months

fig, ax = plt.subplots(figsize=(14, 6))
for col in returns.columns:
    if col == 'SPY':
        continue
    rolling_corr = returns['SPY'].rolling(window).corr(returns[col])
    ax.plot(rolling_corr.index, rolling_corr.values, label=col, linewidth=1.0, alpha=0.8)

ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax.set_title(f'Rolling {window}-Day Correlation with SPY')
ax.set_ylabel('Correlation')
ax.legend(fontsize=8, ncol=3)

for name, (start, end) in STRESS_SCENARIOS.items():
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color='red')

plt.tight_layout()
plt.show()

print('Key observation: Correlations spike during crises (red shading), reducing diversification benefits.')

## 6. Macroeconomic Indicators

In [ ]:
# Plot all macro series
n_series = len(macro.columns)
fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series), sharex=True)

for ax, col in zip(axes, macro.columns):
    data = macro[col].dropna()
    ax.plot(data.index, data.values, linewidth=1.0, color='steelblue')
    ax.set_ylabel(FRED_SERIES.get(col, col), fontsize=9)
    ax.tick_params(labelsize=8)
    
    # Shade stress periods
    for name, (start, end) in STRESS_SCENARIOS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color='red')

axes[0].set_title('Macroeconomic Indicators Over Time')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# Macro summary statistics
macro_summary = macro.describe().T
macro_summary['missing_%'] = macro.isnull().mean() * 100
macro_summary['label'] = [FRED_SERIES.get(c, c) for c in macro_summary.index]
macro_summary[['label', 'count', 'mean', 'std', 'min', 'max', 'missing_%']]

## 7. Stationarity Analysis

In [ ]:
from src.data.preprocessor import test_stationarity, stationarity_report

# Test stationarity of daily returns
print('=== Daily Returns Stationarity (ADF Test) ===')
returns_stationarity = stationarity_report(returns)
returns_stationarity[['stationary', 'p_value', 'test_statistic']]

In [ ]:
# Test stationarity of macro series (levels vs changes)
print('=== Macro Levels Stationarity ===')
macro_levels = stationarity_report(macro)
display(macro_levels[['stationary', 'p_value']])

print('\n=== Macro First Differences Stationarity ===')
macro_diff = macro.diff().dropna()
macro_diff_stat = stationarity_report(macro_diff)
display(macro_diff_stat[['stationary', 'p_value']])

print('\nConclusion: Most macro series are non-stationary in levels but stationary in first differences.')
print('This justifies using changes/lags in our feature engineering.')

## 8. Rolling Volatility & Regime Analysis

In [ ]:
# Rolling volatility (annualized)
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 63-day rolling vol
for col in returns.columns:
    vol = returns[col].rolling(63).std() * np.sqrt(252)
    axes[0].plot(vol.index, vol.values, label=col, linewidth=0.8, alpha=0.8)

axes[0].set_title('63-Day Rolling Annualized Volatility')
axes[0].set_ylabel('Volatility')
axes[0].legend(fontsize=7, ncol=5)

# Average cross-asset volatility (proxy for market regime)
avg_vol = returns.rolling(63).std().mean(axis=1) * np.sqrt(252)
axes[1].fill_between(avg_vol.index, avg_vol.values, alpha=0.5, color='steelblue')
axes[1].set_title('Average Cross-Asset Volatility (Market Regime Proxy)')
axes[1].set_ylabel('Avg Volatility')

# Mark high-vol regimes (above 75th percentile)
threshold = avg_vol.quantile(0.75)
axes[1].axhline(y=threshold, color='red', linestyle='--', linewidth=1, label=f'75th pct: {threshold:.2%}')
axes[1].legend()

for ax in axes:
    for name, (start, end) in STRESS_SCENARIOS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color='red')

plt.tight_layout()
plt.show()

## 9. Macro–Asset Return Relationships

In [ ]:
# Correlate macro changes with next-month asset returns
monthly_returns = returns.resample('ME').sum()
monthly_macro = macro.resample('ME').last().pct_change()

# Align dates
common_idx = monthly_returns.index.intersection(monthly_macro.index)
mr = monthly_returns.loc[common_idx]
mm = monthly_macro.loc[common_idx]

# Lead-lag: do this month's macro changes predict next month's returns?
macro_lead = mm.shift(1)  # lag macro by 1 month
common = macro_lead.dropna().index.intersection(mr.index)
macro_lead = macro_lead.loc[common]
mr_aligned = mr.loc[common]

# Cross-correlation matrix
cross_corr = pd.DataFrame(
    np.corrcoef(macro_lead.T, mr_aligned.T)[:len(macro_lead.columns), len(macro_lead.columns):],
    index=[FRED_SERIES.get(c, c) for c in macro_lead.columns],
    columns=mr_aligned.columns
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(cross_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, vmin=-0.3, vmax=0.3)
ax.set_title('Macro Change (t-1) vs Asset Return (t) — Monthly Correlation')
plt.tight_layout()
plt.show()

print('This matrix motivates our predictive modeling: macro variables with non-zero correlation')
print('contain information about future asset returns.')

## 10. Key Takeaways

1. **Fat tails**: All assets show significant excess kurtosis (Jarque-Bera rejects normality), motivating robust optimization.
2. **Correlation clustering**: Equities are highly correlated; bonds and gold provide diversification — except during crises.
3. **Regime dependence**: Volatility and correlations shift dramatically during stress periods, supporting dynamic allocation.
4. **Macro predictability**: Several macro indicators show non-zero lead-lag correlation with asset returns, justifying the ML approach.
5. **Stationarity**: Returns are stationary; macro levels are not — we must use changes or lags as features.